In [2]:
%pip install pandas pygwalker sqlalchemy pyodbc python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import urllib
from pathlib import Path

import pandas as pd
import pygwalker as pyg
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. Carga de entorno y conexión (Formato Dzib)
# Busca .env en la carpeta actual y en sus directorios padre.
env_file = next(
    (parent / ".env" for parent in [Path.cwd(), *Path.cwd().parents] if (parent / ".env").is_file()),
    None,
)
if env_file:
    load_dotenv(env_file, override=True)

server = os.getenv("DB_SERVER")
database = os.getenv("DB_NAME")
driver = os.getenv("DB_DRIVER", "ODBC Driver 17 for SQL Server")

if not server or not database:
    raise RuntimeError(
        "No se encontraron DB_SERVER y/o DB_NAME. "
        "Configúralas en el archivo .env del proyecto."
    )

params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};"
    "Trusted_Connection=yes;TrustServerCertificate=yes;"
)
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    connect_args={"timeout": 10},
)

# 2. Consumo de Vistas Analíticas
print("📊 Extrayendo datos desde SQL Server...")
df_logistics = pd.read_sql("SELECT * FROM Analytics.vw_Shipping_Efficiency", engine)
df_performance = pd.read_sql("SELECT * FROM Analytics.vw_Category_Performance", engine)

# Validación de contenido (Métrica de éxito: filas recuperadas)
print(f"✅ Filas recuperadas (Logística): {len(df_logistics)}")
print(f"✅ Filas recuperadas (Desempeño): {len(df_performance)}")

# 3. Lanzamiento de Interfaz Interactiva
if not df_logistics.empty:
    print("🚀 Lanzando interfaz interactiva...")
    walker = pyg.walk(df_logistics)
else:
    print("❌ ERROR: El DataFrame de logística está vacío. Revisa la vista en SQL.")

RuntimeError: No se encontraron DB_SERVER y/o DB_NAME. Configúralas en el archivo .env del proyecto.